# Map GWAS SNPs to genes using GFF3 annotations

This notebook annotates SNPs with genes from a genome annotation file. A SNP is mapped to a gene when its chromosome/sequence identifier matches and its base-pair position falls between the gene's start and end coordinates.

The output retains every column from the SNP CSV and adds:

- `Gene`: the overlapping GFF3 gene ID, or multiple IDs separated by semicolons;
- `Gene Count`: the number of overlapping genes; and
- `Annotation Status`: whether the SNP was mapped, intergenic, or located on a sequence absent from the GFF3 file.

## 1. Input requirements

The annotation file must be GFF3-like, with tab-separated fields and gene records in column 3. Coordinates in GFF3 are **1-based and inclusive**. Gene IDs are read from the `ID` attribute by default, with `gene_id` and `Name` used as fallbacks.

The SNP file must be a CSV containing chromosome/sequence and position columns. Their names are configurable below. FastLMM commonly reports these columns as `Chr` and `ChrPos`; other files may use `Chromosome` and `Position`.

> Chromosome or contig identifiers must agree between the two files—for example, `1` will not automatically match `chr1`, and an accession such as `NC_000001.11` will not match `1`. The notebook checks for shared identifiers before mapping.

## 2. Imports and configuration

Replace the three placeholder paths. Set the SNP column names to match the input CSV.

In [ ]:
from pathlib import Path
from urllib.parse import unquote

import numpy as np
import pandas as pd

GFF_FILE = Path("path/to/genome_annotations.gff3")
SNP_FILE = Path("path/to/gwas_snps.csv")
OUTPUT_FILE = Path("path/to/gwas_snps_with_genes.csv")

# Use "Chr" and "ChrPos" for standard FastLMM output.
SNP_CHROMOSOME_COLUMN = "Chromosome"
SNP_POSITION_COLUMN = "Position"

GFF_FEATURE_TYPE = "gene"
GFF_GENE_ID_KEYS = ("ID", "gene_id", "Name")
MULTIPLE_GENE_SEPARATOR = ";"


## 3. Annotation functions

These functions parse gene records from the GFF3 file, validate the SNP coordinates, and map SNPs chromosome by chromosome. The mapping keeps all overlapping genes rather than silently retaining only the first match.

In [ ]:
def normalize_sequence_id(value):
    """Convert a chromosome/contig identifier to a comparable string."""
    sequence_id = str(value).strip()

    # Pandas can read numeric chromosome values as floats (for example, 1.0).
    try:
        numeric_value = float(sequence_id)
        if numeric_value.is_integer():
            return str(int(numeric_value))
    except ValueError:
        pass

    return sequence_id


def parse_gff_attributes(attribute_text):
    """Parse the ninth GFF3 column into a dictionary."""
    attributes = {}
    for field in attribute_text.strip().split(";"):
        field = field.strip()
        if not field:
            continue

        if "=" in field:
            key, value = field.split("=", 1)
        elif " " in field:  # Supports common GTF-style attributes.
            key, value = field.split(" ", 1)
        else:
            continue

        attributes[key.strip()] = unquote(value.strip().strip('\"'))

    return attributes


def load_gene_annotations(
    gff_file,
    feature_type=GFF_FEATURE_TYPE,
    gene_id_keys=GFF_GENE_ID_KEYS,
):
    """Load gene intervals and identifiers from a GFF3/GTF file."""
    gff_file = Path(gff_file)
    annotations = []
    skipped_without_id = 0

    with gff_file.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip() or line.startswith("#"):
                continue

            fields = line.rstrip("\n").split("\t")
            if len(fields) != 9:
                continue
            if fields[2] != feature_type:
                continue

            try:
                start = int(fields[3])
                end = int(fields[4])
            except ValueError as error:
                raise ValueError(
                    f"Invalid coordinates in {gff_file} at line {line_number}."
                ) from error

            if start < 1 or end < start:
                raise ValueError(
                    f"Invalid interval {start}-{end} in {gff_file} at line {line_number}."
                )

            attributes = parse_gff_attributes(fields[8])
            gene_id = next(
                (attributes[key] for key in gene_id_keys if attributes.get(key)),
                None,
            )
            if gene_id is None:
                skipped_without_id += 1
                continue

            annotations.append(
                {
                    "Sequence ID": normalize_sequence_id(fields[0]),
                    "Gene Start": start,
                    "Gene End": end,
                    "Gene ID": gene_id,
                }
            )

    if not annotations:
        raise ValueError(
            f"No '{feature_type}' records with a recognized gene ID were found in {gff_file}."
        )

    annotation_table = (
        pd.DataFrame(annotations)
        .drop_duplicates()
        .sort_values(["Sequence ID", "Gene Start", "Gene End", "Gene ID"])
        .reset_index(drop=True)
    )

    if skipped_without_id:
        print(f"Skipped {skipped_without_id:,} gene record(s) without a recognized ID.")

    return annotation_table


def load_and_validate_snps(snp_file, chromosome_column, position_column):
    """Load a SNP CSV and validate its chromosome and position columns."""
    snps = pd.read_csv(snp_file)
    required_columns = {chromosome_column, position_column}
    missing_columns = required_columns.difference(snps.columns)
    if missing_columns:
        raise KeyError(
            f"SNP file is missing column(s): {sorted(missing_columns)}. "
            f"Available columns: {list(snps.columns)}"
        )

    if snps.empty:
        raise ValueError(f"The SNP file contains no rows: {snp_file}")

    snps = snps.copy()
    if snps[chromosome_column].isna().any():
        bad_rows = snps.index[snps[chromosome_column].isna()].tolist()[:10]
        raise ValueError(f"Missing chromosome identifiers at row index(es): {bad_rows}")

    snps[chromosome_column] = snps[chromosome_column].map(normalize_sequence_id)
    numeric_positions = pd.to_numeric(snps[position_column], errors="coerce")
    invalid_positions = (
        numeric_positions.isna()
        | (numeric_positions < 1)
        | (numeric_positions % 1 != 0)
    )
    if invalid_positions.any():
        bad_rows = snps.index[invalid_positions].tolist()[:10]
        raise ValueError(f"Invalid SNP positions at row index(es): {bad_rows}")

    snps[position_column] = numeric_positions.astype(np.int64)
    return snps


def map_snps_to_genes(
    snps,
    annotations,
    chromosome_column,
    position_column,
    separator=MULTIPLE_GENE_SEPARATOR,
):
    """Annotate each SNP with every gene interval that contains its position."""
    annotated_snps = snps.copy()
    annotated_snps["Gene"] = pd.Series(pd.NA, index=annotated_snps.index, dtype="string")
    annotated_snps["Gene Count"] = 0
    annotated_snps["Annotation Status"] = "Sequence absent from gene annotations"

    annotation_groups = {
        sequence_id: group.sort_values("Gene Start").reset_index(drop=True)
        for sequence_id, group in annotations.groupby("Sequence ID", sort=False)
    }

    for sequence_id, snp_group in annotated_snps.groupby(chromosome_column, sort=False):
        chromosome_annotations = annotation_groups.get(sequence_id)
        if chromosome_annotations is None:
            continue

        ordered_snp_indices = (
            snp_group.sort_values(position_column, kind="stable").index
        )
        genes = list(
            chromosome_annotations[["Gene Start", "Gene End", "Gene ID"]]
            .itertuples(index=False, name=None)
        )

        active_genes = []
        next_gene_index = 0
        for snp_index in ordered_snp_indices:
            position = int(annotated_snps.at[snp_index, position_column])

            while (
                next_gene_index < len(genes)
                and genes[next_gene_index][0] <= position
            ):
                active_genes.append(genes[next_gene_index])
                next_gene_index += 1

            active_genes = [gene for gene in active_genes if gene[1] >= position]
            matching_gene_ids = list(
                dict.fromkeys(gene_id for _, _, gene_id in active_genes)
            )

            if matching_gene_ids:
                annotated_snps.at[snp_index, "Gene"] = separator.join(matching_gene_ids)
                annotated_snps.at[snp_index, "Gene Count"] = len(matching_gene_ids)
                annotated_snps.at[snp_index, "Annotation Status"] = "Mapped to gene"
            else:
                annotated_snps.at[snp_index, "Annotation Status"] = "Intergenic"

    return annotated_snps


## 4. Load and validate the inputs

This cell checks that the files exist, loads the gene and SNP records, and confirms that the files share at least one chromosome or sequence identifier. It also prints identifier examples to help diagnose naming mismatches.

In [ ]:
for input_file in (GFF_FILE, SNP_FILE):
    if not input_file.is_file():
        raise FileNotFoundError(f"Input file not found: {input_file}")

if OUTPUT_FILE.resolve() == SNP_FILE.resolve():
    raise ValueError("OUTPUT_FILE must differ from SNP_FILE to protect the input data.")

annotations = load_gene_annotations(GFF_FILE)
snps = load_and_validate_snps(
    SNP_FILE,
    chromosome_column=SNP_CHROMOSOME_COLUMN,
    position_column=SNP_POSITION_COLUMN,
)

gff_sequences = set(annotations["Sequence ID"])
snp_sequences = set(snps[SNP_CHROMOSOME_COLUMN])
shared_sequences = gff_sequences & snp_sequences

print(f"Gene annotations loaded: {len(annotations):,}")
print(f"SNPs loaded: {len(snps):,}")
print(f"GFF3 sequence IDs (first 10): {sorted(gff_sequences)[:10]}")
print(f"SNP sequence IDs (first 10): {sorted(snp_sequences)[:10]}")
print(f"Shared sequence IDs: {len(shared_sequences):,}")

if not shared_sequences:
    raise ValueError(
        "The SNP and GFF3 files have no matching chromosome/sequence identifiers. "
        "Rename or map the identifiers before annotation."
    )


## 5. Map SNPs to genes and save the annotated CSV

The mapping uses inclusive gene boundaries, so a SNP exactly at a gene's start or end coordinate is counted as overlapping. SNPs outside annotated genes are retained and labeled as intergenic.

In [ ]:
annotated_snps = map_snps_to_genes(
    snps,
    annotations,
    chromosome_column=SNP_CHROMOSOME_COLUMN,
    position_column=SNP_POSITION_COLUMN,
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
annotated_snps.to_csv(OUTPUT_FILE, index=False)

status_counts = annotated_snps["Annotation Status"].value_counts(dropna=False)
print("Annotation summary:")
print(status_counts.to_string())
print(f"\nSNPs overlapping multiple genes: {(annotated_snps['Gene Count'] > 1).sum():,}")
print(f"Annotated results saved to: {OUTPUT_FILE}")
annotated_snps.head()
